In [1]:
import sys
import csv
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag.config import FAQ_FILE

held_out = []
with open(FAQ_FILE, encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if row["split"] == "train":
            continue
        if row["requires_tool"] == "true":
            continue
        if row["source_doc"] == "out_of_scope_policy":
            continue
        held_out.append({"question": row["question"], "language": row["language"], "doc": row["source_doc"]})

counts = {}
for item in held_out:
    counts[item["language"]] = counts.get(item["language"], 0) + 1

print(len(held_out))
print(counts)

1221
{'English': 302, 'Russian': 265, 'Arabic': 239, 'French': 312, 'Arabic-Egyptian': 103}


In [2]:
import time
from my_rag import search as s

totals = {}
hits = {}
blocked = {}
misses = []

start = time.time()
for item in held_out:
    lang = item["language"]
    totals[lang] = totals.get(lang, 0) + 1
    results = s.search(item["question"])
    if results == []:
        blocked[lang] = blocked.get(lang, 0) + 1
    sources = []
    for r in results:
        sources.append(r["source"])
    if item["doc"] in sources:
        hits[lang] = hits.get(lang, 0) + 1
    else:
        misses.append(item)

print("time:", round(time.time() - start), "s for", len(held_out), "questions\n")
for lang in totals:
    recall = round(100 * hits.get(lang, 0) / totals[lang])
    print(lang, "| Recall@5:", recall, "% | wrongly blocked:", blocked.get(lang, 0), "of", totals[lang])

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 14:24:11.208000 90549 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39217.85it/s]


time: 32 s for 1221 questions

English | Recall@5: 100 % | wrongly blocked: 0 of 302
Russian | Recall@5: 95 % | wrongly blocked: 2 of 265
Arabic | Recall@5: 98 % | wrongly blocked: 0 of 239
French | Recall@5: 96 % | wrongly blocked: 1 of 312
Arabic-Egyptian | Recall@5: 79 % | wrongly blocked: 4 of 103


In [3]:
for item in misses:
    if item["language"] != "Arabic-Egyptian":
        continue
    results = s.search(item["question"])
    got = []
    for r in results[:3]:
        got.append(r["source"])
    print(item["question"], "| expected:", item["doc"], "| got:", got)

عايز late check-out، بكام؟ | expected: pricing_and_rates.txt | got: ['check_in_check_out.txt', 'check_in_check_out.txt', 'check_in_check_out.txt']
انتوا في أنهي محافظة؟ | expected: hotel_information.txt | got: []
الخروج المتأخر بكام؟ | expected: pricing_and_rates.txt | got: ['check_in_check_out.txt', 'check_in_check_out.txt', 'check_in_check_out.txt']
التأخير في الخروج ببلاش ولا بفلوس؟ | expected: pricing_and_rates.txt | got: ['check_in_check_out.txt', 'check_in_check_out.txt', 'check_in_check_out.txt']
لو عايز أخرج الساعة 4 هدفع كام زيادة؟ | expected: pricing_and_rates.txt | got: ['check_in_check_out.txt', 'check_in_check_out.txt', 'check_in_check_out.txt']
لازم أدي رقم تليفوني وأنا بحجز؟ | expected: booking_policy.txt | got: ['reservation_security.txt', 'reservation_security.txt', 'reservation_security.txt']
لازم أدفع وأنا بحجز؟ | expected: booking_process.txt | got: ['payment_policy.txt', 'payment_policy.txt', 'booking_policy.txt']
ماوصلنيش تأكيد لحد دلوقتي، أكلم مين؟ | expected: bo

In [4]:
def diff(text):
    q = s._encode(text)
    return float((s._anchor_vecs @ q).max() - (s._vectors @ q).max())

for item in held_out:
    if s.search(item["question"]) == []:
        print(round(diff(item["question"]), 3), "|", item["language"], "|", item["question"])

0.083 | Russian | Какова политика отеля в отношении ресторан отеля?
0.116 | Russian | Не могли бы вы уточнить ресторан отеля?
0.123 | Arabic-Egyptian | انتوا في أنهي محافظة؟
0.07 | Arabic-Egyptian | ماوصلنيش تأكيد لحد دلوقتي، أكلم مين؟
0.073 | Arabic-Egyptian | بياناتي الشخصية في أمان عندكم؟
0.125 | Arabic-Egyptian | النزيل اللي جنبي بيدخن في أوضته والريحة داخلة عندي
0.09 | French | Qu'y a-t-il près de l'hôtel ?


In [5]:
from my_rag.evaluate import load_tests

positives, refuse, faq_out = load_tests()

new_outside = [
    "مين هيكسب الدوري السنة دي؟",
    "اعملي ريجيم أخس بيه 5 كيلو",
    "إيه أحسن موبايل تحت 10 آلاف؟",
    "ترجملي الجملة دي للألماني",
    "عايز أعرف نتيجة الثانوية العامة",
    "اكتبلي بوست لينكدإن عن الشغل",
    "وش أفضل مطعم مندي في الرياض؟",
    "كم سعر تذكرة الطيران من جدة للقاهرة؟",
    "What is the capital of Australia?",
    "Can you help me fix my laptop's wifi driver?",
    "Find me a cheap apartment to rent in Maadi",
    "Quel temps fera-t-il à Paris demain ?",
    "Peux-tu m'aider à écrire une lettre de motivation ?",
    "Сколько стоит iPhone в Москве?",
    "Как приготовить борщ?",
]

d_in = []
for item in held_out:
    d_in.append(diff(item["question"]))
for item in positives:
    d_in.append(diff(item["question"]))

d_old = []
for t in refuse + faq_out:
    if not isinstance(t, str):
        t = t["question"]
    d_old.append(diff(t))

d_new = []
for t in new_outside:
    d_new.append(diff(t))

for m in [0.07, 0.08, 0.09, 0.10, 0.12, 0.15]:
    wrong = 0
    for d in d_in:
        if d > m:
            wrong += 1
    old = 0
    for d in d_old:
        if d > m:
            old += 1
    new = 0
    for d in d_new:
        if d > m:
            new += 1
    print(m, "| correct blocked by mistake:", wrong, "of", len(d_in), "| old outside caught:", old, "of", len(d_old), "| NEW outside caught:", new, "of", len(d_new))

0.07 | correct blocked by mistake: 8 of 1326 | old outside caught: 27 of 27 | NEW outside caught: 10 of 15
0.08 | correct blocked by mistake: 6 of 1326 | old outside caught: 27 of 27 | NEW outside caught: 8 of 15
0.09 | correct blocked by mistake: 4 of 1326 | old outside caught: 25 of 27 | NEW outside caught: 8 of 15
0.1 | correct blocked by mistake: 3 of 1326 | old outside caught: 25 of 27 | NEW outside caught: 8 of 15
0.12 | correct blocked by mistake: 2 of 1326 | old outside caught: 24 of 27 | NEW outside caught: 4 of 15
0.15 | correct blocked by mistake: 0 of 1326 | old outside caught: 21 of 27 | NEW outside caught: 4 of 15
